In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from datetime import timedelta

# --- Configuration ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output4.xlsx'

# Target sensors for feature engineering
target_sensors = ['AssetID_9368','AssetID_9369','AssetID_9370','AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_smart_analysis():
    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return

    # 1. Data Loading & Preprocessing
    df = pd.read_excel(file_path)
    
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    
    df_model = df.dropna(subset=target_sensors).copy()
    
    # 2. Denoising (Moving Average to eliminate operator/sensor glitches)
    # This step ensures we analyze "Real Trends" rather than "Outliers"
    for sensor in target_sensors:
        df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()
    
    df_model = df_model.dropna(subset=[f'{s}_smooth' for s in target_sensors]).copy()
    smooth_cols = [f'{s}_smooth' for s in target_sensors]

    # 3. Feature Scaling
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[smooth_cols])
    
    # 4. Clustering (Behavioral Segmentation)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)

    # 5. Engineering Health Metrics (Degradation Index)
    # Calculating Euclidean distance from the cluster centroid
    distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
    df_model['Degradation_Index'] = distances 

    # 6. Standardized Health Labeling (English Industry Standards)
    def get_health_status(row):
        if row['Degradation_Index'] > 2.5:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 1.5:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

    df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)

    # 7. Filtering for the Last 30 Days
    if 'date' in df_model.columns:
        last_date = df_model['date'].max()
        one_month_ago = last_date - timedelta(days=30)
        final_output = df_model[df_model['date'] >= one_month_ago].copy()
    else:
        final_output = df_model

    # 8. Exporting to Excel
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print("🚀 Analysis completed successfully.")
        print(f"📊 Labels updated to International Engineering Standards.")
        print(f"📁 Output saved: {output_filename}")
    except Exception as e:
        print(f"❌ Error saving file: {e}")

if __name__ == "__main__":
    run_smart_analysis()

c:\Users\pishva_r\.conda\envs\base2\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


🚀 Analysis completed successfully.
📊 Labels updated to International Engineering Standards.
📁 Output saved: outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output4.xlsx
